# Day 38: Build your first autonomous Agent using LangGraph

Welcome to Day 38! Today we are building an **Autonomous Agent** using **LangGraph**. An agent is not just a standard LLM call; it is an LLM that has access to *tools* and can *reason* about when and how to use them.

## Core Theory (Just-in-Time)

### The "Why"
Standard LLMs have cut-off dates for their knowledge and struggle with precise mathematics or executing actions in the real world. By giving them "tools" (like a web search API or a calculator), we empower them to solve complex problems that require external data or exact computation.

### The "How"
We are using **LangGraph**, the modern, state-based way to build agentic loops. Older approaches (like `AgentExecutor` in early LangChain) were often "black boxes" that were hard to debug. LangGraph models the agent as a state machine:
1.  **State:** The memory of the agent (typically a list of messages).
2.  **Nodes:** Python functions that perform work (e.g., calling the LLM, executing a tool).
3.  **Edges:** The logic that connects nodes (e.g., "If the LLM says use a tool, go to the tool node; otherwise, go to the END node").

We will use:
-   `StateGraph` and `MessagesState` to hold the conversation.
-   `ChatOpenAI` (or another LLM) wrapped with `.bind_tools()`.
-   `ToolNode` and `tools_condition` from `langgraph.prebuilt` to handle tool execution and routing.


### AI Security Implication
1.  **Remote Code Execution (RCE):** When using tools like `eval()` for a calculator, malicious prompts can inject Python code. Always sandbox and sanitize inputs.
2.  **PII Data Leaks:** Agents often pass context directly to external APIs. Use a scrubbing proxy or strict PII redaction layer before calling the LLM.
3.  **Prompt Injection:** An attacker might manipulate external search results to issue internal commands to the agent.

## Common Pitfalls in Production
1.  **Infinite Loops:** An agent might get stuck repeatedly calling a tool that fails. Always set a recursion limit when invoking LangGraph (`{"recursion_limit": 10}`).
2.  **Tool Descriptions:** The LLM decides to use a tool based entirely on its docstring/description. If the description is vague, the LLM will hallucinate arguments or call the wrong tool.
3.  **Security (Remote Code Execution):** If you give an agent a Python REPL or `eval()` tool, you must sandbox it! In our calculator example, we strictly whitelist characters to prevent malicious code execution.
4.  **Deprecated Frameworks:** Avoid `ToolExecutor` and `ToolInvocation` (from LangGraph 0.1.x) and `AgentExecutor` (from LangChain). Use `ToolNode` and `tools_condition`.


## Reference Links
- [LangGraph Official Documentation](https://python.langchain.com/docs/langgraph)
- [Building Autonomous Agents](https://lilianweng.github.io/posts/2023-06-23-agent/)



## 1. Setup and Dependencies

Let's install the necessary packages and set up our environment.

In [ ]:
# uv pip install langchain langchain-openai langgraph duckduckgo-search ddgs

import os
import re
from typing import List, Annotated, Sequence

# We will use DuckDuckGo for search, which is free and requires no API key.
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage, AIMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

# Set a dummy key if not present (for local testing without a real OpenAI key)
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = "sk-dummy-key"


## 2. Basic Implementation

Here we isolate the core concept of an Agent using LangGraph with minimal boilerplate. We define a single tool and a basic state.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

@tool
def basic_calculator(expression: str) -> str:
    """Evaluates a math expression."""
    try:
        # Simplified for basic example
        return str(eval(expression, {"__builtins__": None}, {}))
    except Exception as e:
        return f"Error: {e}"

class BasicState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# 1. Initialize LLM with tools
basic_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
basic_tools = [basic_calculator]
basic_llm_with_tools = basic_llm.bind_tools(basic_tools)

# 2. Graph definition
def basic_reasoner(state: BasicState):
    try:
        return {"messages": [basic_llm_with_tools.invoke(state["messages"])]}
    except Exception as e:
        return {"messages": [AIMessage(content=f"API Error: {e}")]}

builder = StateGraph(BasicState)
builder.add_node("reasoner", basic_reasoner)
builder.add_node("tools", ToolNode(tools=basic_tools))
builder.add_edge(START, "reasoner")
builder.add_conditional_edges("reasoner", tools_condition)
builder.add_edge("tools", "reasoner")
basic_app = builder.compile()

# 3. Run
# res = basic_app.invoke({"messages": [HumanMessage(content="What is 2 * 10?")]}, {"recursion_limit": 10})
# print(res["messages"][-1].content)


## 3. Medium Implementation

This implementation emphasizes clean OOP, state management, and clear interactions between the LLM and its tools.

In [ ]:
from typing import List, Any

class AgentSystem:
    def __init__(self, tools: List[Any], model_name: str = "gpt-4o-mini"):
        self.llm = ChatOpenAI(model=model_name, temperature=0)
        self.tools = tools
        self.llm_with_tools = self.llm.bind_tools(tools)
        self.app = self._build_graph()

    def _reasoner(self, state: BasicState):
        try:
            return {"messages": [self.llm_with_tools.invoke(state["messages"])]}
        except Exception as e:
            return {"messages": [AIMessage(content=f"API Error: {e}")]}

    def _build_graph(self):
        graph = StateGraph(BasicState)
        graph.add_node("reasoner", self._reasoner)
        graph.add_node("tools", ToolNode(tools=self.tools))
        graph.add_edge(START, "reasoner")
        graph.add_conditional_edges("reasoner", tools_condition)
        graph.add_edge("tools", "reasoner")
        return graph.compile()

    def run(self, query: str):
        print(f"\n--- Running Medium Agent: {query} ---")
        state = {"messages": [HumanMessage(content=query)]}
        try:
            for event in self.app.stream(state, {"recursion_limit": 10}):
                for node, update in event.items():
                    msg = update["messages"][-1]
                    if isinstance(msg, AIMessage) and not msg.tool_calls:
                        print(f"🤖 Answer: {msg.content}")
                    elif isinstance(msg, ToolMessage):
                        print(f"🛠️ Tool {msg.name} Result: {msg.content}")
        except Exception as e:
            print(f"Agent Error: {e}")

# Usage
# agent = AgentSystem(tools=[basic_calculator])
# agent.run("What is 50 / 5?")


## 4. Advanced Implementation

A production-grade implementation featuring strict type hinting, docstrings, secure fallback mechanisms, and robust error handling.

In [ ]:
import os
import logging

from typing import Sequence, Optional, Dict, Any, Annotated
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage
from pydantic import BaseModel, Field
from langchain_community.tools import DuckDuckGoSearchRun

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class SecureCalculatorInput(BaseModel):
    expression: str = Field(..., description="The mathematical expression to evaluate.")

@tool(args_schema=SecureCalculatorInput)
def secure_calculator(expression: str) -> str:
    """
    Securely evaluates mathematical expressions.
    Only allows numbers and basic operators.
    """
    allowed_chars = set("0123456789+-*/(). ")
    if not all(char in allowed_chars for char in expression):
        logger.warning(f"Blocked potentially malicious input: {expression}")
        return "Error: Invalid characters detected. Calculation blocked for security."
    try:
        return str(eval(expression, {"__builtins__": None}, {}))
    except Exception as e:
        logger.error(f"Calculation error: {e}")
        return "Error: Invalid mathematical expression."

class ProductionAgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

class ProductionAgent:
    """Production-ready LangGraph Agent."""
    
    def __init__(self, tools: Sequence[Any], model_name: str = "gpt-4o"):
        self.tools = tools
        api_key = os.environ.get("OPENAI_API_KEY")
        if not api_key:
            logger.warning("OPENAI_API_KEY is not set. Falling back to dummy key.")
            os.environ["OPENAI_API_KEY"] = "sk-dummy"
            
        self.llm = ChatOpenAI(model=model_name, temperature=0)
        self.llm_with_tools = self.llm.bind_tools(tools)
        self.app = self._compile_graph()
        
    def _compile_graph(self):
        def reasoner_node(state: ProductionAgentState) -> Dict[str, list]:
            try:
                response = self.llm_with_tools.invoke(state["messages"])
                return {"messages": [response]}
            except Exception as e:
                logger.error(f"LLM API error: {e}")
                return {"messages": [AIMessage(content=f"System Fallback: Service unavailable ({e})")]}

        graph = StateGraph(ProductionAgentState)
        graph.add_node("reasoner", reasoner_node)
        graph.add_node("tools", ToolNode(tools=self.tools))
        graph.add_edge(START, "reasoner")
        graph.add_conditional_edges("reasoner", tools_condition)
        graph.add_edge("tools", "reasoner")
        return graph.compile()

    def run(self, query: str) -> Optional[str]:
        logger.info(f"Processing query: {query}")
        state = {"messages": [HumanMessage(content=query)]}
        try:
            result = self.app.invoke(state, config={"recursion_limit": 10})
            return result["messages"][-1].content
        except Exception as e:
            logger.error(f"Agent execution failed: {e}")
            return None

# Production Usage:
# search_tool = DuckDuckGoSearchRun()
# prod_agent = ProductionAgent(tools=[secure_calculator, search_tool])
# print(prod_agent.run("What is 100 * 4?"))



## 5. Practical Lab / Homework

Your task is to add a new tool to the agent.

**Task:**
1.  Create a new `@tool` called `get_current_weather` that takes a `city: str` as input.
2.  Implement the tool using a mock dictionary. E.g., if the city is "New York", return "75°F and Sunny". If the city is "London", return "60°F and Rainy". Otherwise, return "Weather unknown".
3.  Add this new tool to the `tools` list.
4.  Re-run the agent with the query: `"What is the weather in London right now?"`


**Bonus:** Record a brief async video walkthrough (e.g., Loom) explaining your design decisions and how you managed state.


In [ ]:
@tool
def get_current_weather(city: str) -> str:
    """
    Gets the current weather for a given city.
    """
    city_weather_data = {
        "new york": "75°F and Sunny",
        "london": "60°F and Rainy",
        "tokyo": "80°F and Clear",
        "paris": "65°F and Cloudy"
    }
    return city_weather_data.get(city.lower(), "Weather unknown for this location.")
# We use the clean OOP implementation from Section 3
# which correctly re-binds the LLM to the new tools
all_tools = [basic_calculator, get_current_weather]
weather_agent = AgentSystem(tools=all_tools)
weather_agent.run("What is the weather in London right now?")
